# Workshop: Introducción a Generative AI en Oracle y Creación de Agentes con LangChain

En este workshop aprenderás, de forma práctica y paso a paso, cómo usar los servicios de IA generativa de Oracle Cloud y cómo construir un agente inteligente con LangChain.

A lo largo del notebook trabajaremos sobre un tema central que servirá como hilo conductor para:

- consumir modelos de lenguaje alojados en Oracle,

- generar texto de manera controlada,

- y crear un agente capaz de buscar información y razonar usando herramientas.

El ejercicio evoluciona progresivamente: comenzamos con inferencias simples y terminamos con un agente que combina modelos, tools y buenas prácticas de orquestación.

### Objetivos

- Integrar OCI Generative AI desde Python

- Ejecutar inferencias controladas con modelos de lenguaje

- Construir un agente con LangChain que use herramientas (búsqueda, SQL, RAG)

- Entender cómo diseñar flujos de trabajo con agentes

> 💡 **Tip:** este notebook está diseñado para ser práctico y paso a paso. Podrás copiar, ejecutar y modificar el código para experimentar con los conceptos que vamos a explicar.

¡Vamos a empezar!

## Parte 1: Construcción de un agente con una herramienta de búsqueda web

### Importación de librerías

In [ ]:
import json
import os
import re
from typing import Dict, List

import oci
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.chat_models import ChatOCIGenAI
from langchain_community.embeddings import OCIGenAIEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from oci.auth.signers import get_resource_principals_signer
from oci.config import from_file
from oci.generative_ai import GenerativeAiClient
from tavily import TavilyClient



## Registro y configuración de Tavily

 Nuestro agente necesita acceso a la web para acceder a las últimas noticias del tema que le hemos indicado, para esto, es necesario configurar una herramienta que le permita a nuestro agente navegar por portales de noticias, por eso usaremos [Tavily](https://www.tavily.com/).

#### 🪪 Registro en Tavily (paso a paso)

1) Abre **https://app.tavily.com/home** y haz clic en **Sign Up**. Verifica tu correo electrónico para activar la cuenta. ![image1](https://github.com/martingg92/deepdive_workshop_oci_2026/blob/main/images/tavily_signup.png?raw=True)
2) Inicia sesión: en la **página principal** verás tu **API Key**. Haz **Copy** para copiarla. ![image2](https://github.com/martingg92/deepdive_workshop_oci_2026/blob/main/images/api_key.png?raw=True)


### Configuración de las variables de Tavily

In [ ]:
# Pega la API Key de Tavily aquí
TAVILY_API_KEY = "tvly-..."

In [ ]:
# No edites los bloques de código que contengan un statement de assert. 
# Estos bloques validan que las variables se hayan configurado correctamente. 
assert TAVILY_API_KEY.startswith("tvly-"), "Por favor, pega tu API Key de Tavily en la variable TAVILY_API_KEY"

## Creación de una herramienta

Para la construcción de nuestro agente necesitamos definir tools o herramientas.
El siguiente codigo como se denota crea una lista con las paginas oficiales del noticias para poder con tavily extraer informacion relevante de acuerdo al topico de noticia a buscar

In [ ]:

DOMAINS = [
    "www.goal.com",
    "www.soccerblog.com",
    "www.101greatgoals.com",
    "worldsoccertalk.com",
    "www.semana.com",
    "www.elpais.com.co"
]

@tool
def search_in_internet(pregunta:str="¿Qué es la inflación?") -> str:
    """Busca información en internet. Usa esta herramienta para buscar información actualizada en internet que no esté en tu base de conocimientos."""

    client = TavilyClient(TAVILY_API_KEY)
    response = client.search(
    query=pregunta,
    include_domains=DOMAINS,
    #topic="news",
    #days=45,
    #max_results=10
    )
    return response

## Configuración de la autenticación del SDK de OCI

A continuación será necesario autenticarte, los pasos para obtener tus credenciales se encuentran [aquí](https://github.com/valentinafeve/oracle-labs-ai/blob/main/utils/Creaci%C3%B3n%20de%20credenciales.md)

In [ ]:
# crea carpeta y permisos
!mkdir -p /Workspace/.oci

In [ ]:
"""
Reemplazaremos ocid1.user.oc1.. por el ocid del usuario mostrado en pantalla
Reemplazaremos fingerprint por el figerprint mostrado en pantalla
Reemplazaremos tenancy por el figerprint mostrado en pantalla
Reemplazaremos region por la region mostrado en pantalla
"""
user_ocid    = "ocid1.user.oc1.."  # ocid1.user.oc1..xxxxx
fingerprint  = "a1:2b:"  # xx:xx:xx:xx:xx:xx:xx:xx:xx:xx:xx:xx:xx:xx:xx:xx
tenancy_ocid = "ocid1.tenancy.oc1.."  # ocid1.tenancy.oc1..xxxxx
region       = "eu-frankfurt-1"  # ej: us-ashburn-1

In [ ]:
import pathlib

oci_dir = pathlib.Path("/Workspace")
oci_dir.mkdir(parents=True, exist_ok=True)

config_path = oci_dir / "config"
config_path.write_text(f"""[DEFAULT]
user={user_ocid}
fingerprint={fingerprint}
tenancy={tenancy_ocid}
region={region}
key_file=/Workspace/key.pem
""")
config_path.chmod(0o600)

print(f"📝 Archivo de configuración fue creado en: {config_path}")
print(f"")
print(f"   user       → {user_ocid}")
print(f"   fingerprint→ {fingerprint}")
print(f"   tenancy    → {tenancy_ocid}")
print(f"   region     → {region}")
print(f"   key_file   → /Workspace/key.pem")
print(f"")
print(f"✅ Listo. Permisos: {oct(config_path.stat().st_mode)[-3:]}")

In [ ]:
import pathlib, shutil

workspace = pathlib.Path("/Workspace")
pem_files = list(workspace.glob("*.pem"))

if not pem_files:
    raise FileNotFoundError("No se encontró ningún archivo .pem en /Workspace/")

src = pem_files[0]
dest = workspace / "key.pem"

print(f"🔍 Archivo encontrado:  {src.name}")
print(f"🔑 Será usado como key: {dest}")

if src != dest:
    shutil.copy2(src, dest)
    print(f"📁 Guardado en:         {dest}")
else:
    print(f"📁 Ya se encuentra en:  {dest}")

dest.chmod(0o600)
print(f"✅ Listo. Permisos:     {oct(dest.stat().st_mode)[-3:]}")

## Instanciación de un cliente de OCI Gen AI

Una vez configuradas las credenciales, debemos instanciar un cliente de OCI Gen AI para acceder al servicio de IA generativa.

### Configuración de Variables

In [ ]:
#carga a memoria la configuracion de OCI para enviarla al API de Generative AI
config = from_file("/Workspace/config")

In [ ]:
assert config['key_file'] == '/Workspace/key.pem'

🚨 Descomenta únicamente la línea que corresponda a tu región en la cual esta tu tenant y ejecuta la celda.

In [ ]:
# Descomenta únicamente la línea que corresponda a tu región en la cual esta tu tenant
#REGION = "sa-saopaulo-1"
#REGION = "us-chicago-1"
#REGION = "uk-london-1"
#REGION = "eu-frankfurt-1"
#REGION = "ap-osaka-1"
#REGION = "us-ashburn-1"

match REGION:
    case "us-chicago-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyayjawvuonfkw2ua4bob4rlnnlhs522pafbglivtwlfzta"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "sa-saopaulo-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.sa-saopaulo-1.amaaaaaask7dceyarsn4m6k3aqvvgatida3omyprlcs3alrwcuusblru4jaa"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "uk-london-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.uk-london-1.amaaaaaask7dceyach2dyu6g5w5ocnvbkto2g76wxitj3rpddplsqoxqh2lq"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "eu-frankfurt-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.eu-frankfurt-1.amaaaaaask7dceya4tdabclcsqbc3yj2mozvvqoq5ccmliv3354hfu3mx6bq"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "ap-osaka-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.ap-osaka-1.amaaaaaask7dceyaei4b6vhy7rgsqzuqet4ndnnun4fhxco2tfzusslg35wa"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "us-ashburn-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.iad.amaaaaaask7dceyah6tjdejjashngznsylutuhhvufukzb2g2ls54g2flsfq"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"


In [ ]:
# No ajustes este bloque de código. Este bloque es solo para validar si la region se configuró como una variable
assert REGION in ["sa-saopaulo-1", "us-chicago-1", "uk-london-1", "eu-frankfurt-1", "ap-osaka-1"], "Por favor, descomenta la línea que corresponda a tu región"

⚠️Aquí debes pegar el OCID de tu compartimento⚠️
Encuentra el OCID de tu compartimento en la consola de Oracle Cloud, en la sección de Compartments https://cloud.oracle.com/identity/compartments

In [ ]:
# Aquí debes pegar el OCID de tu compartimento
# Encuentra el OCID de tu compartimento en la consola de Oracle Cloud, en la sección de Compartments https://cloud.oracle.com/identity/compartments
COMPARTMENT_ID = "ocid1.compartment.oc1.."

In [ ]:
SERVICE_ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"


Ahora puedes validar y listar todos los modelos generativos en la región, para tu conocimiento.

In [ ]:
genai = GenerativeAiClient(config=config)
models = genai.list_models(
    compartment_id=COMPARTMENT_ID,
    capability=["CHAT"],
    lifecycle_state="ACTIVE"
).data.items
assert models, "No hay modelos CHAT visibles en el compartimento. Revisa permisos/compartimento."

print("Modelos CHAT disponibles:")
for model in models:
    print(f"{model.display_name} by {model.vendor}")

Hasta este punto, hemos completado las configuraciones necesarias para utilizar los servicios de IA. A continuación, realizaremos una inferencia sencilla con una pregunta simple para obtener la respuesta del modelo.

In [ ]:
# Aquí se crea el cliente de inferencia
inf = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=config,
    service_endpoint=SERVICE_ENDPOINT
)

# Este prompt corresponde a la pregunta que se le hará al modelo
user_input = "¿Quién es el técnico de la selección Colombia?"

# Aquí se construye el request
content = oci.generative_ai_inference.models.TextContent(text=user_input)
message = oci.generative_ai_inference.models.Message(role="USER", content=[content])

chat_request = oci.generative_ai_inference.models.GenericChatRequest(
    api_format=oci.generative_ai_inference.models.BaseChatRequest.API_FORMAT_GENERIC,
    messages=[message],
    max_tokens=15,
    temperature=0.1,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    seed=42
)

chat_detail = oci.generative_ai_inference.models.ChatDetails(
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_OCID),
    chat_request=chat_request,
    compartment_id=COMPARTMENT_ID,
)

# Aquí se realiza la llamada al modelo
resp = inf.chat(chat_detail)

# Aquí se procesa el resultado
choices = resp.data.chat_response.choices
response_text = choices[0].message.content[0].text if choices else "No se generó respuesta."
print(json.dumps({"response": response_text}, indent=2, ensure_ascii=False))

### ❓ Preguntas

- ¿cuál es tu región?
- ¿cuántos modelos hay en tu región?
- ¿qué parámetro del modelo define el tamaño de la respuesta?
- Haz un análisis de la pregunta y la respuesta del modelo

## 🤖 Creación del Agente LangChain

LangChain es una librería que facilita la creación de aplicaciones impulsadas por modelos de lenguaje de gran tamaño (LLMs). En Oracle Cloud Infrastructure (OCI), LangChain se integra con los servicios de IA generativa, permitiendo construir agentes que interactúan con modelos de lenguaje alojados en OCI.
En las siguientes secciones, exploraremos cómo crear un agente utilizando LangChain y OCI.

Un agente, está compuesto de un flujo que onvolucra un modelo de lenguaje capaz de utilizar herramientas y generar respuestas a partir de los resultados de las herramientas.

![imagen](https://github.com/martingg92/deepdive_workshop_oci_2026/blob/main/images/agent_flow.png?raw=True)

In [ ]:
# Configura tu endpoint y compartimento
#ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"

llm = ChatOCIGenAI(
  model_id=MODEL_OCID,
  service_endpoint=SERVICE_ENDPOINT,
  compartment_id=COMPARTMENT_ID,
  provider="meta",
  model_kwargs={
    "temperature": 0.3, 
    "max_tokens": 800,   
    "top_p": 0.8,  
    "frequency_penalty": 0,
    "presence_penalty": 0,
  },
  auth_type="API_KEY",
  auth_profile="DEFAULT",
    auth_file_location="/Workspace/config"
)

tools = [
    # Aquí se deben agregar las herramientas que se quieran usar en el agente, por ejemplo: search_in_internet
    search_in_internet
]

react_prompt_template = """

Eres un asistente inteligente diseñado para ayudar a los usuarios a resolver preguntas y tareas de forma clara, útil y confiable, independientemente del dominio (noticias, finanzas, tecnología, educación, salud, u otros).

Tu objetivo es comprender la intención del usuario, razonar sobre el problema y entregar una respuesta clara.

Herramientas disponibles:
{tools}

Usa EXACTAMENTE este formato:

Question: la pregunta a responder
Thought: explica qué harás
Action: una de [{tool_names}]
Action Input: el input para la acción (o "" si no aplica)
Observation: resultado de la acción
Thought: analiza y sintetiza
Final Answer: respuesta clara y útil en español 

Comienza.

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_prompt_template)
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=30,
    stream_runnable=False,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)
pregunta = "¿Quién es el técnico de la selección Colombia?"
respuesta = agent_executor.invoke({"input": pregunta})


In [ ]:
respuesta

### ❓ Preguntas

- ¿Qué función cumple la herramienta o tool que se incluyó en el agente?

- ¿Cuál es la diferencia entre la respuesta del modelo anterior con generative ai y la respuesta del agente con una tool?

Ahora vamos a ver como tomamos la respuesta generada por el agente y normalízala a texto plano antes de usarla o mostrarla. La idea es extraer únicamente el contenido útil. Por ejemplo, el mensaje final, evitando incluir objetos, metadatos o estructuras internas del framework. Esto ayuda a mantener un formato consistente, reducir errores de procesamiento y presentar una salida más limpia y segura.

In [ ]:
# 1) Toma la salida del agente y conviértela a texto de forma segura
def _to_text(x):
    if isinstance(x, dict):
        # LangChain AgentExecutor suele devolver {"output": "..."}
        return x.get("output") or json.dumps(x, ensure_ascii=False, indent=2)
    return str(x)
insumos = respuesta   # <--- usa la variable 'respuesta' que ya tienes del agente
# 2) Prompt de análisis/síntesis para Fiestas Patrias (Chile)
analysis_prompt = f"""
Eres un asistente analítico cuyo objetivo es comprender información proporcionada y transformarla en una respuesta clara, estructurada y útil para el usuario.

Responde exclusivamente en español.

A continuación se entregan los únicos insumos que puedes utilizar.
Pueden contener texto libre, listas, tablas o estructuras como JSON, y pueden estar incompletos.
{insumos}
---
### TAREAS

Responde de forma directa y clara a la pregunta planteada.

Extrae y organiza la información relevante solo si está disponible en los insumos, priorizando lo más importante.

Identifica hechos, acciones, fechas, impactos, actores o medidas relevantes cuando existan.

Si hay diferencias o inconsistencias entre los insumos, menciónalas brevemente y de forma neutral.

Si falta información clave, indícalo explícitamente.
---
### FORMATO (Markdown)
## Resumen
Síntesis breve (3–5 líneas) con los puntos más relevantes para entender la situación.

## Detalles relevantes (si hay información)
Elemento / tema: descripción clara
Información clave
Condiciones o contexto
Observaciones relevantes
Enlace o referencia (si existe)

## Recomendaciones o acciones (si hay información)
Lista clara y concreta de acciones, sugerencias o conclusiones derivadas de los insumos.

## Fuentes
Lista de referencias o enlaces o solo si aparecen explícitamente en los insumos.
Si no hay referencias, escribe: No disponible.
---
### REGLAS
-  No inventes datos, cifras, fechas, actores ni enlaces.
- Si una sección no tiene información suficiente, indica: No disponible.
- No hagas suposiciones ni completes vacíos con conocimiento previo.
- Mantén un tono claro, neutral y orientado a facilitar la comprensión.
"""
# 3) Invoca la LLM (no streaming) y muestra en Markdown
analysis_response = llm.invoke(analysis_prompt)
print(analysis_response.content)


### ❓ Preguntas
- ¿Qué sucede si agrego una quinta tarea indicandole al agente que debe hacer una predicción futurista?
- ¿El modelo conoce los detalles de la conexión a la base de conocimiento?
- ¿El modelo comprende la implementación de las herramientas que posee?
- ¿Qué puedes hacer para que el modelo evite responder información de sus herramientas internas?

------